<a href="https://colab.research.google.com/github/dharvi120/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*



## Unit of Analysis

One row in the source table (`fact_content_daily_performance`) represents the daily performance of one content page (`content_hash_id`) belonging to one client (`client_hash_id`) on one reporting date (`report_date`).

For the machine learning dataset, the prediction unit will be one content page for one client at a prediction point. Daily observations will later be aggregated into a historical feature window so that each training example represents a single page ready for refresh prioritization.

## Time Window

Feature Window:
Historical performance before the prediction date (e.g., previous 90 days).

Prediction Point:
The date on which refresh priority is calculated.

Label Window:
Future performance after the prediction date. This window will be used to determine whether the page experienced a decline that justifies prioritization for content refresh.

The feature and label windows must not overlap to avoid data leakage.

**-- Verify dataset coverage**


SELECT

    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(*) AS total_rows

FROM fact_content_daily_performance;

**-- Verify grain on a representative day**

SELECT

    content_hash_id,
    client_hash_id,
    report_date,
    COUNT(*) AS cnt

FROM fact_content_daily_performance

WHERE report_date = DATE '2025-01-27'

GROUP BY

    content_hash_id,
    client_hash_id,
    report_date
HAVING COUNT(*) > 1;

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



| Field | Bucket | Why |
|--------|---------|-----|
| report_date | Context | Defines when the observation occurred. |
| client_hash_id | Context | Identifies the client. Not a predictive signal by itself. |
| content_hash_id | Context | Identifies the content page. |
| gsc_impressions | Feature | Search visibility before prediction. |
| gsc_clicks | Feature | Historical search traffic. |
| gsc_avg_position | Feature | Historical ranking signal. |
| ga4_pageviews | Feature | Historical user activity. |
| ga4_sessions | Feature | Historical engagement signal. |
| ga4_users | Feature | Historical audience size. |
| ga4_engaged_sessions | Feature | User engagement quality. |
| ga4_total_engagement_sec | Feature | Engagement duration. |
| sessions_organic | Feature | Organic acquisition signal. |
| sessions_direct | Feature | Direct traffic behaviour. |
| sessions_referral | Feature | Referral traffic behaviour. |
| sessions_social | Feature | Social traffic behaviour. |
| sessions_paid | Feature | Paid traffic behaviour. |
| sessions_ai | Feature | AI referral activity. |
| ai_chatgpt | Feature | AI traffic source. |
| ai_gemini | Feature | AI traffic source. |
| ai_claude | Feature | AI traffic source. |
| ai_perplexity | Feature | AI traffic source. |
| ai_copilot | Feature | AI traffic source. |
| ai_meta | Feature | AI traffic source. |
| ai_other | Feature | Other AI referral traffic. |
| scroll_events | Feature | User engagement behaviour. |
| Refresh Priority Score (future) | Label | Target to be predicted. Computed from future performance, not available at prediction time. |
| Future clicks / impressions / position | Excluded | Future information would leak the answer into training. |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

SELECT COUNT(*) AS total_rows

FROM fact_content_daily_performance;

SELECT

MIN(report_date),

MAX(report_date)

FROM fact_content_daily_performance;

SELECT

content_hash_id,

client_hash_id,

report_date,

COUNT(*) AS cnt

FROM fact_content_daily_performance

WHERE report_date='2025-01-27'

GROUP BY

content_hash_id,

client_hash_id,

report_date

HAVING COUNT(*)>1;

SELECT

SUM(CASE WHEN ga4_data_available = FALSE THEN 1 ELSE 0 END) AS ga4_missing,

SUM(CASE WHEN gsc_data_available = FALSE THEN 1 ELSE 0 END) AS gsc_missing

FROM fact_content_daily_performance;

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

SELECT

SUM(CASE WHEN ga4_data_available = FALSE THEN 1 ELSE 0 END) AS ga4_missing,

SUM(CASE WHEN gsc_data_available = FALSE THEN 1 ELSE 0 END) AS gsc_missing

FROM fact_content_daily_performance;



- The source table contains daily observations rather than model-ready page-level records, so aggregation is required before training.
- GA4 metrics may be unavailable for some observations (`ga4_data_available = FALSE`). Missing analytics data should not automatically be interpreted as zero user activity.
- Google Search Console metrics may also be unavailable for some observations (`gsc_data_available = FALSE`).
- The available data spans from 2025-01-27 to 2026-06-30, limiting the maximum historical and future windows that can be constructed.
- Historical feature windows and future label windows must remain separate to prevent target leakage.
- This data contract describes the source data. Additional feature engineering (rolling averages, trends, CTR, etc.) will be performed in later stages of the project.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.